# 02 - How Modern RAG Works

## Definition
Modern RAG is a staged pipeline that separates retrieval quality from generation quality.

## Theory
Dense embeddings map semantics to vectors; vector search returns nearest contexts; prompt assembly grounds response synthesis.

## Motivation
Debugging is easier when each stage has independent metrics and diagnostics.

## Architecture
User Query -> Embedding -> Vector Search -> Retrieved Context -> Prompt Construction -> Generation -> Answer.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [1]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [2]:
bundle = runner.ingest_dataset()
len(bundle['documents']), len(bundle['queries']), bundle['summary']


(20233,
 11873,
 {'num_documents': 20233,
  'num_queries': 11873,
  'num_unique_titles': 477,
  'doc_length_mean': 739.6197795680324,
  'doc_length_median': 679.0,
  'doc_length_p95': 1315.0,
  'question_length_mean': 59.441169038996044,
  'question_length_p95': 98.0,
  'answer_length_mean': 20.916160593792174,
  'answerable_ratio': 0.4992840899519919,
  'unanswerable_ratio': 0.5007159100480081,
  'query_split_counts': {'validation': 11873}})

In [3]:
chunking = runner.run_chunking(bundle['documents'][:260], strategy=ChunkingStrategy.RECURSIVE)
runner.index_chunks(chunking.chunks, reset=True)

sample_query = bundle['queries'][0].query
result = runner.pipeline.answer(sample_query, top_k=6)

{
    'query': sample_query,
    'answer_preview': result.answer[:280],
    'abstained': result.abstained,
    'num_retrieved': len(result.retrieved_chunks),
    'citations': result.citations,
}


{'query': 'In what country is Normandy located?',
 'answer_preview': 'I cannot find enough evidence in retrieved documents. The provided context mentions Lorraine [4] but does not specify the current or historical location of that region, nor does it contain information regarding Normandy.',
 'abstained': False,
 'num_retrieved': 6,
 'citations': ['[1] doc=doc_b40979b83d6b2898 title=Frédéric_Chopin score=0.667',
  '[2] doc=doc_2da6c0f543267811 title=Frédéric_Chopin score=0.652',
  '[3] doc=doc_9b6db7c346c4db51 title=Frédéric_Chopin score=0.642',
  '[4] doc=doc_92a3f137376f4dc8 title=Frédéric_Chopin score=0.640',
  '[5] doc=doc_dccc38f6e70ca130 title=Frédéric_Chopin score=0.638',
  '[6] doc=doc_931c45bb7dc9b834 title=Frédéric_Chopin score=0.634']}

## Code Explanation
This walkthrough runs a minimal retrieve-augment-generate loop and inspects retrieved chunks before trusting output text.